In [ ]:
# Punto 4 - Método de Monte Carlo para calcular π
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Para que siempre se obtengan resultados similares
np.random.seed(42)

# ==========================================================
# MÉTODO DE MONTE CARLO
# ==========================================================

def montecarlo_pi(N):
    """
    Calcula una aproximación de π utilizando
    N puntos aleatorios.
    """

    x = np.random.uniform(-1, 1, N)
    y = np.random.uniform(-1, 1, N)

    dentro = (x**2 + y**2) <= 1

    return 4 * np.sum(dentro) / N


# ==========================================================
# MODELOS DE CONVERGENCIA
# ==========================================================

def modelo_algebraico(N, A, alpha):
    """
    Modelo:
        f(N)=A/N^alpha
    """
    return A / (N**alpha)


def modelo_exponencial(N, A, lamb):
    """
    Modelo:
        f(N)=A exp(-lambda N)
    """
    return A * np.exp(-lamb * N)


# ==========================================================
# COEFICIENTE DE DETERMINACIÓN
# ==========================================================

def calcular_r2(y, y_ajuste):

    ss_res = np.sum((y - y_ajuste) ** 2)

    ss_tot = np.sum((y - np.mean(y)) ** 2)

    return 1 - ss_res / ss_tot


# ==========================================================
# AJUSTE NO LINEAL
# ==========================================================

def ajustar_modelo(N, f):

    resultados = {}

    # ------------------------
    # Ajuste algebraico
    # ------------------------

    try:

        parametros_alg, _ = curve_fit(
            modelo_algebraico,
            N,
            f,
            p0=[1, 0.5],
            maxfev=10000
        )

        ajuste_alg = modelo_algebraico(N, *parametros_alg)

        r2_alg = calcular_r2(f, ajuste_alg)

    except:

        parametros_alg = None
        ajuste_alg = None
        r2_alg = -np.inf

    resultados["algebraico"] = {
        "parametros": parametros_alg,
        "ajuste": ajuste_alg,
        "r2": r2_alg
    }

    # ------------------------
    # Ajuste exponencial
    # ------------------------

    try:

        parametros_exp, _ = curve_fit(
            modelo_exponencial,
            N,
            f,
            p0=[1, 1e-6],
            maxfev=10000
        )

        ajuste_exp = modelo_exponencial(N, *parametros_exp)

        r2_exp = calcular_r2(f, ajuste_exp)

    except:

        parametros_exp = None
        ajuste_exp = None
        r2_exp = -np.inf

    resultados["exponencial"] = {
        "parametros": parametros_exp,
        "ajuste": ajuste_exp,
        "r2": r2_exp
    }

    return resultados

    # ==========================================================
# BÚSQUEDA DEL MEJOR π∞
# ==========================================================

def buscar_mejor_pi(N, pi_N):

    # Valores candidatos para π∞
    pi_inf_posibles = np.linspace(3.13, 3.15, 2000)

    r2_alg = []
    r2_exp = []

    parametros_alg = []
    parametros_exp = []

    for pi_inf in pi_inf_posibles:

        # Función pedida en el enunciado
        f = np.abs(pi_inf - pi_N)

        resultados = ajustar_modelo(N, f)

        r2_alg.append(resultados["algebraico"]["r2"])
        r2_exp.append(resultados["exponencial"]["r2"])

        parametros_alg.append(resultados["algebraico"]["parametros"])
        parametros_exp.append(resultados["exponencial"]["parametros"])

    r2_alg = np.array(r2_alg)
    r2_exp = np.array(r2_exp)

    # Mejor ajuste algebraico
    indice_alg = np.argmax(r2_alg)

    # Mejor ajuste exponencial
    indice_exp = np.argmax(r2_exp)

    # Comparación de modelos
    if r2_alg[indice_alg] >= r2_exp[indice_exp]:

        mejor_modelo = "Algebraico"

        mejor_pi = pi_inf_posibles[indice_alg]

        mejor_r2 = r2_alg[indice_alg]

        mejor_parametros = parametros_alg[indice_alg]

        mejor_ajuste = "algebraico"

    else:

        mejor_modelo = "Exponencial"

        mejor_pi = pi_inf_posibles[indice_exp]

        mejor_r2 = r2_exp[indice_exp]

        mejor_parametros = parametros_exp[indice_exp]

        mejor_ajuste = "exponencial"

    return {

        "modelo": mejor_modelo,

        "pi_infinito": mejor_pi,

        "r2": mejor_r2,

        "parametros": mejor_parametros,

        "tipo_ajuste": mejor_ajuste,

        "pi_inf_posibles": pi_inf_posibles,

        "r2_algebraico": r2_alg,

        "r2_exponencial": r2_exp

    }


# ==========================================================
# FUNCIÓN PARA GRAFICAR
# ==========================================================

def graficar_pi(N, pi_N):

    plt.figure(figsize=(8,5))

    plt.plot(
        N,
        pi_N,
        'o-',
        label='Monte Carlo'
    )

    plt.axhline(
        np.pi,
        color='red',
        linestyle='--',
        label='Valor real'
    )

    plt.xscale("log")

    plt.xlabel("Número de puntos (N)")

    plt.ylabel(r"$\pi(N)$")

    plt.title("Estimación de π mediante Monte Carlo")

    plt.grid(True)

    plt.legend()

    plt.tight_layout()

    plt.savefig("MonteCarloPi.png",dpi=300)

    plt.show()


# ==========================================================
# ERROR ABSOLUTO
# ==========================================================

def graficar_error(N, pi_N):

    error = np.abs(np.pi-pi_N)

    plt.figure(figsize=(8,5))

    plt.loglog(
        N,
        error,
        'o-'
    )

    plt.xlabel("Número de puntos (N)")

    plt.ylabel(r"$|\pi-\pi(N)|$")

    plt.title("Error absoluto")

    plt.grid(True)

    plt.tight_layout()

    plt.savefig("ErrorMonteCarlo.png",dpi=300)

    plt.show()

    # ==========================================================
# PROGRAMA PRINCIPAL
# ==========================================================

# Valores de N (cantidad de puntos aleatorios)

N = np.unique(np.logspace(2, 6, 40, dtype=int))

# Calcular π(N)

pi_N = np.array([montecarlo_pi(n) for n in N])

# Graficar π(N)

graficar_pi(N, pi_N)

# Graficar error

graficar_error(N, pi_N)

# ==========================================================
# BÚSQUEDA DEL MEJOR π∞
# ==========================================================

resultado = buscar_mejor_pi(N, pi_N)

pi_inf = resultado["pi_infinito"]

modelo = resultado["modelo"]

r2 = resultado["r2"]

parametros = resultado["parametros"]

tipo = resultado["tipo_ajuste"]

# ==========================================================
# CALCULAR f(π∞)
# ==========================================================

f = np.abs(pi_inf - pi_N)

if tipo == "algebraico":

    ajuste = modelo_algebraico(N, *parametros)

else:

    ajuste = modelo_exponencial(N, *parametros)

# ==========================================================
# GRÁFICA DE f(π∞)
# ==========================================================

plt.figure(figsize=(8,5))

plt.loglog(
    N,
    f,
    'o',
    label='Datos'
)

plt.loglog(
    N,
    ajuste,
    '-',
    linewidth=2,
    label='Ajuste'
)

plt.xlabel("Número de puntos (N)")

plt.ylabel(r"$f(\pi_\infty)=|\pi_\infty-\pi(N)|$")

plt.title("Ajuste de la función f")

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.savefig("FuncionPiInfinito.png", dpi=300)

plt.show()

# ==========================================================
# GRÁFICA DE R²
# ==========================================================

plt.figure(figsize=(8,5))

plt.plot(
    resultado["pi_inf_posibles"],
    resultado["r2_algebraico"],
    label="Modelo algebraico"
)

plt.plot(
    resultado["pi_inf_posibles"],
    resultado["r2_exponencial"],
    label="Modelo exponencial"
)

plt.axvline(
    pi_inf,
    color='red',
    linestyle='--',
    label=r'Mejor $\pi_\infty$'
)

plt.xlabel(r"$\pi_\infty$")

plt.ylabel(r"$R^2$")

plt.title(r"Coeficiente de determinación $R^2(\pi_\infty)$")

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.savefig("R2MonteCarlo.png", dpi=300)

plt.show()

# ==========================================================
# RESULTADOS FINALES
# ==========================================================

error_abs = abs(np.pi - pi_inf)

error_rel = error_abs / np.pi

error_porcentual = error_rel * 100

print("\n" + "="*60)
print("RESULTADOS")
print("="*60)

print(f"Valor real de π          : {np.pi:.10f}")

print(f"π∞ estimado              : {pi_inf:.10f}")

print(f"Modelo de convergencia   : {modelo}")

print(f"Coeficiente R²           : {r2:.8f}")

print(f"Error absoluto           : {error_abs:.10e}")

print(f"Error relativo           : {error_rel:.10e}")

print(f"Error porcentual         : {error_porcentual:.8f} %")

print("="*60)